# PNN and chromatic-dispersion equalization: BER/EVM study

This notebook generates BER-versus-distance and EVM-versus-distance at 0 dBm launch power, then BER/EVM-versus-launch-power at a 1000 km link. It compares baseline coherent reception, conventional CDC, PNN without CDC, PNN jointly trained with conventional CDC, and PNN jointly trained with joint CDC. Each PNN strategy is trained using both BER and EVM objectives, with independent test sequences.

## Theoretical framing

A coherent optical receiver mixes the received optical field with a local-oscillator (LO) field and recovers in-phase and quadrature components. This retains information in both amplitude and phase and supports digital compensation, including chromatic-dispersion compensation.

Laser phase noise is commonly modeled as a Wiener process. For linewidth `Δν` and sample interval `Δt`, the phase-increment variance is `2π Δν Δt`; increasing linewidth therefore increases phase diffusion over a fixed observation interval. The three phase-noise contributions relevant to this study are: (1) transmitter-laser phase noise, (2) receiver LO phase noise, and (3) nonlinear phase noise associated with propagation and ASE/Kerr interaction. The simulator represents the first two with a correlated TX/LO model (`ρ = 0.99`). The power sweep enables Kerr, but the ideal EDFA adds no ASE; consequently, this is not a full ASE-driven nonlinear phase-noise model.

A fractionally-spaced equalizer (FSE) samples or places transversal taps at intervals shorter than one symbol period. Gitlin, Weinstein and coauthors describe how a fractionally-spaced transversal equalizer can reduce sensitivity to sampling phase and realize an optimum linear receiver under the stated assumptions. Here, at 10 GBaud, `Ts = 100 ps`; the modeled PNN tap spacing of 25 ps is `Ts/4`. The PNN is an optical tapped-delay-line FIR whose complex tap weights are formed by amplitude and phase controls. The fractional spacing is an analogy in tap spacing, not proof of equivalence to an adaptive electrical FSE: the model uses a finite number of taps, constrained amplitude weights, ideal phase controls and simplified optical boundaries/detection.

References: [Gitlin, Meadors & Weinstein (1981), *Fractionally-Spaced Equalization: An Improved Digital Transversal Equalizer*](https://doi.org/10.1002/j.1538-7305.1981.tb00240.x); [Bell Labs publication record](https://www.nokia.com/bell-labs/publications-and-media/publications/fractionally-spaced-equalization-an-improved-digital-transversal-equalizer/).

## Simulation conventions and limitations

The distance sweep uses a linear fiber model at 0 dBm launch power. The power sweep uses SSFM with Kerr at a fixed distance of 1000 km and launch powers from −20 to +20 dBm in 2 dB steps. In both sweeps the link has 50 km spans with ideal EDFAs that compensate span loss. No ASE or frequency offset is included. A constant carrier-phase offset is estimated from known training symbols and applied to the independent test frame for BER; time-varying CPE/BPS and timing recovery are not modeled. Joint CDC inverts the modeled linear fiber dispersion and PNN field response; it does not invert Kerr. All conclusions from the Kerr power sweep must be interpreted with that limitation. BER is estimated on a finite test frame and plotted with a floor, so floor-valued points are not necessarily zero BER.

The script saves the numerical table, metadata, BER/EVM figures and constellation figures. The PSO setup is configurable near the top of the script. A full run is computationally intensive because a separate PNN is optimized for each sweep point, linewidth, objective and equalizer family.

In [1]:
from pathlib import Path
import sys
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'scripts' / 'generate_pnn_cdc_analysis.py').is_file():
    PROJECT_ROOT = PROJECT_ROOT.parent
assert (PROJECT_ROOT / 'scripts' / 'generate_pnn_cdc_analysis.py').is_file()
get_ipython().run_line_magic('run', str(PROJECT_ROOT / 'scripts' / 'generate_pnn_cdc_analysis.py'))

Completed distance=50, linewidth=0 kHz
Completed distance=100, linewidth=0 kHz
Completed distance=150, linewidth=0 kHz
Completed distance=200, linewidth=0 kHz
Completed distance=250, linewidth=0 kHz
Completed distance=300, linewidth=0 kHz
Completed distance=350, linewidth=0 kHz
Completed distance=400, linewidth=0 kHz
Completed distance=450, linewidth=0 kHz
Completed distance=500, linewidth=0 kHz
Completed distance=550, linewidth=0 kHz
Completed distance=600, linewidth=0 kHz
Completed distance=650, linewidth=0 kHz
Completed distance=700, linewidth=0 kHz
Completed distance=750, linewidth=0 kHz
Completed distance=800, linewidth=0 kHz
Completed distance=850, linewidth=0 kHz
Completed distance=900, linewidth=0 kHz
Completed distance=950, linewidth=0 kHz
Completed distance=1000, linewidth=0 kHz
Completed distance=50, linewidth=100 kHz
Completed distance=100, linewidth=100 kHz
Completed distance=150, linewidth=100 kHz
Completed distance=200, linewidth=100 kHz
Completed distance=250, linewidth

OSError: [Errno 22] Invalid argument: 'D:\\gits\\PHOTONICS_2026\\Codigos\\python\\pnn_coerente\\artifacts\\1000.0\\figures\\distance_ber_ber_conventional_cdc.png'

## Generated artifacts

- `artifacts/reports/pnn_cdc_ber_evm_sweeps.csv`: BER, EVM, compared-bit count, objective value and PNN weights per test point.
- `artifacts/reports/pnn_cdc_ber_evm_sweeps_metadata.json`: simulation configuration and BER plot floor.
- `artifacts/figures/distance_*.png` and `power_*.png`: BER and EVM curves, separated by training objective and conventional/joint CDC.
- `artifacts/figures/*constellations*.png`: 300 km constellations for the distance sweep and 0 dBm constellations for the power sweep, for all three linewidths and four strategies.

The plotted four strategies are the two no-PNN baselines, PNN without CDC, and the jointly trained PNN with the selected CDC family. The conventional-CDC and joint-CDC panels share the same baseline curves.